# Lab 1: Chat completions

Send a chat request, stream a response, and keep conversation history in your application. Then inspect output limits and retry behavior.

**Prerequisites:** [setup](../docs/00_prework.md) and a team key. Allow about 25 minutes. The long-context and thinking examples are optional explorations and may take longer.

[Notebook index](README.md) · [API reference](../docs/inference.md) · [Troubleshooting](../docs/troubleshooting.md#inference)

These examples use the workshop service configuration. Confirm aliases and quotas with the organizers. Event details used as sample data are fictional.


## Step 1: Set up your environment

The setup cell loads the workshop `.env` and configures the client explicitly. It does not print the key. The workshop settings override any other provider settings in the notebook process.

In [1]:
# Install the packages used in this notebook.
%pip install -q "openai>=3,<4" python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Run Jupyter from the workshop directory or its notebooks directory.
workshop_dir = Path.cwd().resolve()
if workshop_dir.name == "notebooks":
    workshop_dir = workshop_dir.parent
env_file = workshop_dir / ".env"
if not (workshop_dir / ".env.example").is_file() or not env_file.is_file():
    raise RuntimeError("Open Jupyter from the workshop directory after completing docs/00_prework.md.")
load_dotenv(env_file, override=True)

base_url = os.environ.get("OPENAI_BASE_URL", "").strip().rstrip("/")
api_key = os.environ.get("OPENAI_API_KEY", "").strip()
if not base_url or not api_key or api_key == "hk_paste_your_team_key_here":
    raise RuntimeError("Set OPENAI_BASE_URL and your team key in the workshop .env file.")
print("Endpoint:", base_url)
print("Team key loaded. Key value is hidden.")
client = OpenAI(base_url=base_url, api_key=api_key, timeout=60.0)


Endpoint: https://ws.akamai-workshops.com/v1
Team key loaded. Key value is hidden.


## Request reference

`POST /v1/chat/completions` accepts the model alias, a message list, and options such as `max_tokens`, `temperature`, and `stream`.

The response includes `choices[0].message`, `choices[0].finish_reason`, and token usage. Start with this short call:

In [3]:
response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Say hello in five words."}],
    max_tokens=50,
)
print(response.choices[0].message.content)

Hello there, nice to meet you.


## Step 2: Send a request

A message has a role and content. `system` gives application instructions, `user` provides the request, and `assistant` represents prior model responses. The alias `hackathon-chat` selects the workshop chat model.

Inspect the response object, then read the answer and usage fields.

In [4]:
response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "In two sentences, what is a hackathon?"}],
)

# The whole object, exactly as the server returned it.
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-9ce79e3af06e7e8e",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "A hackathon is a short, intense event where participants, typically software developers and designers, collaborate to create a project or solve a specific problem within a set timeframe. It fosters rapid innovation and teamwork, often culminating in the demonstration of a working prototype or solution.",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "reasoning": null
      },
      "stop_reason": null,
      "token_ids": null,
      "routed_experts": null
    }
  ],
  "created": 1790957433,
  "model": "Qwen/Qwen3.8-27B-FP8",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": null,
  "system_fingerprint": "vllm-0.30.0-tp4-34cf4502",
  "usage": {
    "com

In [5]:
print("Answer:       ", response.choices[0].message.content)
print("Finish reason:", response.choices[0].finish_reason)
print("Tokens:       ", response.usage.prompt_tokens, "in,", response.usage.completion_tokens, "out")

Answer:        A hackathon is a short, intense event where participants, typically software developers and designers, collaborate to create a project or solve a specific problem within a set timeframe. It fosters rapid innovation and teamwork, often culminating in the demonstration of a working prototype or solution.
Finish reason: stop
Tokens:        22 in, 55 out


Inspect `model` and `usage` while developing. A response may report a backing model identifier rather than the alias you sent. Keep aliases in your requests unless the organizers specify otherwise.

## Step 3: Stream the response

With `stream=True`, the client receives chunks as generation proceeds. Collect nonempty `delta.content` values. Latency depends on the request and service load; measure it in your application rather than assuming a fixed response time.

In [6]:
stream = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Give me three names for a study-group app, one per line."}],
    stream=True,
    stream_options={"include_usage": True},
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
    if chunk.usage:  # Only the final chunk has this.
        print("\n\nTokens:", chunk.usage.completion_tokens, "out")

Brainwave
StudySync
GroupGenius

Tokens: 10 out


## Step 4: Keep conversation history

Chat requests are independent. The API does not automatically attach earlier messages to a later request. Run the following two calls and inspect whether the second has enough information to answer.

In [7]:
first = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "My name is Ada. Reply with just the word Noted."}],
)
print("Turn 1:", first.choices[0].message.content)

# A second request with no history. The server has already forgotten turn 1.
second = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "What is my name? Answer in one short sentence."}],
)
print("Turn 2 without history:", second.choices[0].message.content)

Turn 1: Noted
Turn 2 without history: I do not know your name.


Store the messages in your application and include the relevant history in the next request. This example keeps it in a Python list. Use [PostgreSQL](../docs/02_memory.md) to retain it across application restarts.

In [8]:
history = [
    {"role": "system", "content": "You are a concise assistant for a hackathon team."},
    {"role": "user", "content": "My name is Ada. Reply with just the word Noted."},
]
reply = client.chat.completions.create(model="hackathon-chat", messages=history)
history.append({"role": "assistant", "content": reply.choices[0].message.content})

history.append({"role": "user", "content": "What is my name? Answer in one short sentence."})
reply = client.chat.completions.create(model="hackathon-chat", messages=history)
print("Turn 2 with history:", reply.choices[0].message.content)

Turn 2 with history: Your name is Ada.


## Step 5: Control output length

The supplied deployment defaults to 512 output tokens and caps output at 4,096. Set a suitable `max_tokens` value and inspect `finish_reason`. A value of `length` means generation stopped at a limit.

The second example requests more than the documented cap to inspect this deployment's behavior. Use values within the limit in your application.

In [9]:
short = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Explain how a hash map works."}],
    max_tokens=20,
)
print(short.choices[0].message.content)
print("\nFinish reason:", short.choices[0].finish_reason)

# How a Hash Map Works

A **hash map** (also called a hash table or dictionary

Finish reason: length


In [10]:
big = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Reply with the single word OK."}],
    max_tokens=100000,  # Above the cap. The server lowers it to 4096 instead of rejecting the request.
)
print(big.choices[0].message.content, "| finish reason:", big.choices[0].finish_reason)

OK | finish reason: stop


Lower `temperature` values reduce sampling variation but do not guarantee identical results. Use schema validation when your application needs a particular output structure.

In [11]:
for temperature in (0.0, 1.0):
    r = client.chat.completions.create(
        model="hackathon-chat",
        messages=[{"role": "user", "content": "Name one color. One word only."}],
        temperature=temperature,
        max_tokens=5,
    )
    print(f"temperature={temperature}: {r.choices[0].message.content.strip()}")

temperature=0.0: Blue
temperature=1.0: Blue


## Step 6: Work with long documents

The supplied chat context window is 32,768 tokens, including input and output. Leave room for the response, system instructions, and tool definitions. Word or page counts are only rough estimates.

The next cell builds synthetic text containing one specific fact. Observe whether the model finds it. For larger collections, use [retrieval](02_embeddings_and_retrieval.ipynb) to select relevant passages.

In [12]:
filler = ("The committee reviewed the quarterly logistics report and noted that shipping volumes in the "
          "northern region remained steady while maintenance costs rose modestly. Several members "
          "proposed revisiting the vendor contracts next spring. ")
pages = [filler * 12 for _ in range(15)]
pages[9] += " IMPORTANT: the warehouse door code is 7391-ORCHID. "
document = "\n\n".join(f"Page {i + 1}\n{page}" for i, page in enumerate(pages))
print("Words in document:", len(document.split()))

response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": document + "\n\nQuestion: what is the warehouse door code? Answer with just the code."}],
    max_tokens=30,
)
print("Answer:       ", response.choices[0].message.content.strip())
print("Prompt tokens:", response.usage.prompt_tokens)

Words in document: 5797
Answer:        7391-ORCHID
Prompt tokens: 6606


Trim or summarize history before it exceeds the context limit. Do not assume the service will safely shorten a request for you.

## Step 7: Try thinking mode

This optional deployment feature requests additional reasoning before the final answer. It can use more output tokens and take longer than a normal request. The example reads both the deployment-specific reasoning field and final content.

Skip this step during the live quickstart. The request uses a longer timeout for this exercise.

In [13]:
import time

start = time.time()
response = client.with_options(timeout=240.0).chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "A bat and a ball cost $1.10 together. The bat costs $1.00 more than the ball. What does the ball cost?"}],
    max_tokens=1500,
    extra_body={"chat_template_kwargs": {"enable_thinking": True}},
)
message = response.choices[0].message
extra = message.model_extra or {}  # Not a standard OpenAI field, so the SDK keeps it in model_extra.
reasoning = extra.get("reasoning") or extra.get("reasoning_content") or ""  # Deployment-specific field names.
answer = message.content or ""

print("Reasoning (first 400 characters):", reasoning[:400].strip(), "...\n")
print("Answer:", answer.strip() or "(cut off before the answer; raise max_tokens)")
print(f"\nFinish reason: {response.choices[0].finish_reason} | {response.usage.completion_tokens} tokens in {time.time() - start:.0f}s")

Reasoning (first 400 characters): We need answer classic riddle. User: bat and ball cost $1.10 together. Bat costs $1.00 more than ball. What does ball cost? Need answer 5 cents, not 10. Need maybe explain briefly. Need final. Ensure no hidden trap. Let's solve: let ball = b, bat = b+1.00, total 2b+1=1.10 => 2b=0.10 => b=0.05. Final concise. ...

Answer: The ball costs **$0.05**.

If the ball is $0.05, the bat is $1.05, and together they cost $1.10.

Finish reason: stop | 153 tokens in 4s


If `finish_reason` is `length`, inspect whether the output budget was consumed before a complete answer. Compare answer quality and elapsed time before enabling this mode for a user-facing feature.

## Step 8: Handle rate limits

All members and applications using the team key share its quota. The [API reference](../docs/inference.md#deployment-limits-to-confirm) lists the values from the supplied deployment.

The helper below disables SDK retries and handles `429` explicitly. It waits for the service's delay header, adds a small random delay, and limits attempts.

In [14]:
import random
import time
import openai

client = OpenAI(base_url=base_url, api_key=api_key, timeout=60.0, max_retries=0)  # Handle retries ourselves so the behavior is visible.

def ask(prompt, attempts=5):
    for attempt in range(attempts):
        try:
            r = client.chat.completions.create(
                model="hackathon-chat",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=60,
            )
            return r.choices[0].message.content
        except openai.RateLimitError as e:
            if attempt == attempts - 1:
                raise
            wait = float(e.response.headers.get("retry-after") or e.response.headers.get("x-ratelimit-reset") or 5) + random.uniform(0, 2)
            print(f"Rate limited; waiting {wait:.0f}s before attempt {attempt + 2}")
            time.sleep(wait)
    raise RuntimeError("Gave up after repeated rate limits")

print(ask("One sentence of advice for a hackathon team."))

client = OpenAI(base_url=base_url, api_key=api_key, timeout=60.0)  # Restore SDK retries.

Focus on building a working Minimum Viable Product that solves a core problem, rather than a broken feature-rich dream.


## Step 9: Check API compatibility

The supplied endpoint exposes chat completions, embeddings, and image generation. Model listing is not part of its documented API. The next cell inspects that unsupported request; your application should use the published aliases.

In [19]:
try:
    client.models.list()
except openai.APIStatusError as e:
    print("GET /v1/models is not exposed. Status:", e.status_code)

The supplied service also excludes the Responses API, audio, image editing, and image input to chat. See [tools and JSON](04_tools_and_structured_output.ipynb) for the supported application features.

## Validation

Run a short request, inspect its answer, and check that usage was returned. Model wording can vary; the assertion checks that a response exists.

In [20]:
check = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Reply with the single word READY."}],
    max_tokens=5,
)
assert "READY" in check.choices[0].message.content.upper(), check.choices[0].message.content
assert check.usage.total_tokens > 0
print("Lab 1 complete.")

Lab 1 complete.


## Clean up and continue

No cloud resources were created. Clear notebook outputs before sharing.

For a deployed chat app, use the [Linode lab](../docs/03_body.md). For persistent history, use [PostgreSQL](../docs/02_memory.md). Choose [retrieval](02_embeddings_and_retrieval.ipynb) if answers need your documents.